# Capítulo 6: Transformers y Large Language Models (LLMs)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caracena/apunte-analitica-textual/blob/main/capitulos/clase6-transformers-llms.ipynb)

## Objetivos de aprendizaje

- Comprender la arquitectura Transformer y el mecanismo de atención.
- Entender el concepto de transferencia de aprendizaje y fine-tuning.
- Conocer los principales modelos pre-entrenados (BERT, GPT).
- Utilizar LLMs para tareas de generación, resumen y modificación de texto.

## 6.1 De Word2Vec a Transformers

### Limitaciones de Word2Vec

- **Representación estática**: "banco" tiene el mismo vector sin importar si es un asiento o una entidad financiera.
- **Sin contexto**: No captura el significado según el uso en la oración.

### La evolución

```
BoW/TF-IDF → Word2Vec/GloVe → ELMo → Transformer → BERT/GPT → LLMs
(disperso)    (denso estático)  (contextual)  (atención)  (pre-entrenado)  (masivo)
```

Los **Transformers** (Vaswani et al., 2017) introducen representaciones **contextuales**: cada palabra obtiene un vector diferente según su contexto.

## 6.2 La arquitectura Transformer

### Componentes clave

1. **Self-Attention (Auto-atención)**: Permite a cada token "atender" a todos los demás tokens de la secuencia, capturando dependencias a cualquier distancia.

2. **Multi-Head Attention**: Múltiples cabezas de atención en paralelo capturan diferentes tipos de relaciones.

3. **Positional Encoding**: Inyecta información sobre la posición de cada token en la secuencia.

4. **Feed-Forward Network**: Red neuronal que procesa cada posición independientemente.

### Fórmula de atención

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Donde:
- $Q$ (Query), $K$ (Key), $V$ (Value) son proyecciones lineales de la entrada.
- $d_k$ es la dimensión de las claves (factor de escala).

### Encoder vs. Decoder

| Componente | Modelo | Tarea principal |
|-----------|--------|-----------------|
| **Encoder** | BERT, RoBERTa | Comprensión (clasificación, NER, QA) |
| **Decoder** | GPT, LLaMA | Generación de texto |
| **Encoder-Decoder** | T5, BART | Traducción, resumen |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Implementación simplificada de Self-Attention
def self_attention(X, d_k=None):
    """Calcula self-attention simplificada (sin proyecciones aprendidas)."""
    if d_k is None:
        d_k = X.shape[1]
    # Q, K, V son la misma entrada (self-attention simplificada)
    scores = X @ X.T / np.sqrt(d_k)  # (n x n)
    weights = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)  # softmax
    output = weights @ X  # (n x d)
    return output, weights

# Ejemplo: 4 tokens con embeddings de dimensión 3
np.random.seed(42)
tokens = ["el", "gato", "persigue", "ratón"]
X = np.random.randn(4, 3)

output, weights = self_attention(X)

# Visualizar pesos de atención
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(weights, cmap='Blues')
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, fontsize=12)
ax.set_yticklabels(tokens, fontsize=12)
ax.set_title('Pesos de Self-Attention', fontsize=14)

# Agregar valores
for i in range(len(tokens)):
    for j in range(len(tokens)):
        ax.text(j, i, f'{weights[i,j]:.2f}', ha='center', va='center', fontsize=11)

plt.colorbar(im)
plt.tight_layout()
plt.show()

## 6.3 BERT: Bidirectional Encoder Representations from Transformers

**BERT** (Devlin et al., 2019) es un modelo Transformer pre-entrenado bidireccionalmente. Esto significa que para representar una palabra, considera tanto el contexto izquierdo como derecho simultáneamente.

### Pre-entrenamiento

BERT se pre-entrena con dos tareas:
1. **Masked Language Model (MLM)**: Se enmascara el 15% de los tokens y el modelo aprende a predecirlos.
2. **Next Sentence Prediction (NSP)**: El modelo predice si dos oraciones son consecutivas.

### Fine-tuning

Después del pre-entrenamiento, BERT se puede ajustar (*fine-tune*) para tareas específicas con relativamente pocos datos etiquetados.

In [ ]:
from transformers import pipeline

# Ejemplo 1: Fill-mask (tarea MLM de BERT)
fill_mask = pipeline('fill-mask', model='dccuchile/bert-base-spanish-wwm-uncased')

resultado = fill_mask('La inteligencia [MASK] está transformando la industria.')

print("Fill-mask: 'La inteligencia [MASK] está transformando la industria.'\n")
for r in resultado[:3]:
    print(f"  {r['token_str']:15s} (score: {r['score']:.4f})")

In [ ]:
# Ejemplo 2: Análisis de sentimiento con modelo pre-entrenado
clasificador = pipeline(
    'sentiment-analysis',
    model='nlptown/bert-base-multilingual-uncased-sentiment'
)

textos = [
    "Este producto es excelente, me encantó",
    "Pésima calidad, no lo recomiendo",
    "Es un producto aceptable, nada especial"
]

print("Análisis de sentimiento:\n")
for texto in textos:
    resultado = clasificador(texto)[0]
    print(f"  '{texto}'")
    print(f"    → {resultado['label']} (confianza: {resultado['score']:.3f})\n")

## 6.4 Transferencia de aprendizaje y Fine-tuning

La **transferencia de aprendizaje** permite usar conocimiento aprendido en una tarea general (pre-entrenamiento en grandes corpus) y adaptarlo a una tarea específica.

### Proceso

```
1. Pre-entrenamiento (corpus masivo, tarea general)
   → Modelo aprende representaciones generales del lenguaje

2. Fine-tuning (dataset específico, tarea objetivo)
   → Se ajustan los pesos del modelo para la tarea concreta
```

### Ventajas

- Requiere muchos menos datos etiquetados.
- Converge más rápido.
- Obtiene mejores resultados que entrenar desde cero.

In [ ]:
# Ejemplo de extracción de embeddings contextuales con BERT
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained('dccuchile/bert-base-spanish-wwm-uncased')
model = AutoModel.from_pretrained('dccuchile/bert-base-spanish-wwm-uncased')

# Obtener embeddings contextuales
texto = "El banco ofrece préstamos a bajo interés"
inputs = tokenizer(texto, return_tensors='pt', padding=True, truncation=True)

with torch.no_grad():
    outputs = model(**inputs)

# outputs.last_hidden_state: embeddings contextuales de cada token
embeddings = outputs.last_hidden_state
print(f"Texto: '{texto}'")
print(f"Tokens: {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])}")
print(f"Shape de embeddings: {embeddings.shape}")
print(f"  → {embeddings.shape[1]} tokens x {embeddings.shape[2]} dimensiones")

### Ejemplo de Fine tuning

In [ ]:
import datasets
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

data_hf = datasets.load_dataset('caracena/lista_espera_sample')

In [ ]:
# 1. Load Tokenizer and Model
# Using a BERT model pre-trained for Spanish
model_name = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

In [ ]:
# 2. Prepare the Dataset for Hugging Face Transformers
data_hf = datasets.load_dataset('caracena/lista_espera_sample')

train_hf_dataset = data_hf["train"]
test_hf_dataset = data_hf["test"]

In [ ]:
# 3. Tokenize the datasets
def tokenize_function(examples):
    # Truncation and padding are important for BERT input
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train_dataset = train_hf_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_hf_dataset.map(tokenize_function, batched=True)

In [ ]:
# 4. Define compute_metrics function for evaluation
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
# 5. Define TrainingArguments
training_args = TrainingArguments(
    output_dir="./results_bert",      # Directory to save model checkpoints and logs
    num_train_epochs=3,              # Number of training epochs
    per_device_train_batch_size=128,  # Batch size for training on each device
    per_device_eval_batch_size=128,   # Batch size for evaluation on each device
    logging_strategy="epoch",               # Log at the end of each epoch
    eval_strategy="epoch",     # Evaluate at the end of each epoch
    save_strategy="epoch",           # Save checkpoint at the end of each epoch
    load_best_model_at_end=True,     # Load the best model after training
    metric_for_best_model="f1",      # Metric to use to compare models during evaluation
    report_to="none",                # Disable integration with W&B, MLflow, etc.
    seed=42,                         # Seed for reproducible results
)

In [ ]:
# 6. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset, # Use test set for evaluation during training
    compute_metrics=compute_metrics,
)

In [ ]:
# 7. Train the model
print("Starting BERT model training...")
trainer.train()
print("BERT model training finished.")

In [ ]:
# 8. Evaluate the model on the test set and display results
print("\nEvaluating BERT model on the test set...")
predictions_bert = trainer.predict(tokenized_test_dataset)
preds_bert = np.argmax(predictions_bert.predictions, axis=-1)

print("\nClassification Report for BERT model:")
print(classification_report(test_hf_dataset['label'], preds_bert))

# Display Confusion Matrix
cm_bert = confusion_matrix(test_hf_dataset['label'], preds_bert)
disp_bert = ConfusionMatrixDisplay(confusion_matrix=cm_bert, display_labels=['Class 0', 'Class 1'])
disp_bert.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix for BERT Model (Fine-tuned on Preprocessed Text)")
plt.show()

## 6.5 Large Language Models (LLMs)

Los **LLMs** son modelos de lenguaje masivos (billones de parámetros) entrenados en enormes corpus de texto. Representan el estado del arte en NLP.

### Características

- **Escala**: Desde millones hasta billones de parámetros.
- **Capacidades emergentes**: A mayor escala, surgen capacidades nuevas (razonamiento, planificación).
- **Few-shot / Zero-shot**: Pueden realizar tareas con pocos o ningún ejemplo.
- **Instrucciones**: Modelos ajustados para seguir instrucciones humanas.

### Modelos destacados

| Modelo | Organización | Parámetros | Tipo |
|--------|-------------|------------|------|
| GPT-4 | OpenAI | ~1.8T (estimado) | Cerrado |
| Claude | Anthropic | No publicado | Cerrado |
| LLaMA 3 | Meta | 8B - 405B | Abierto |
| Gemini | Google | No publicado | Cerrado |
| Mistral | Mistral AI | 7B - 8x22B | Abierto |

In [ ]:
# Ejemplo: Generación de texto con un modelo generativo
generador = pipeline(
    'text-generation',
    model='datificate/gpt2-small-spanish',
    max_new_tokens=50
)

prompt = "La inteligencia artificial en la medicina permite"
resultado = generador(prompt, num_return_sequences=1)

print("Generación de texto:")
print(f"  Prompt: '{prompt}'")
print(f"  Generado: '{resultado[0]['generated_text']}'")

## 6.6 Prompting: El arte de comunicarse con LLMs

El **prompting** es la técnica de diseñar instrucciones efectivas para obtener la respuesta deseada de un LLM.

### Tipos de prompting

| Técnica | Descripción | Ejemplo |
|---------|-------------|---------|
| **Zero-shot** | Sin ejemplos | "Clasifica el sentimiento: ..." |
| **Few-shot** | Con ejemplos | "Positivo: me encanta. Negativo: lo odio. Clasifica: ..." |
| **Chain-of-thought** | Razonamiento paso a paso | "Piensa paso a paso..." |
| **Instrucciones** | Instrucciones explícitas | "Eres un experto en..." |

## Ejemplos prompting

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch

# Reutilizamos el modelo de generación GPT-2 en español que ya fue usado antes
# Aunque este modelo no es un LLM "instruction-tuned" como Gemma, 
# servirá para demostrar los conceptos básicos de prompting.
print("Cargando modelo LLM para demostración de prompting (gpt2-small-spanish)...")
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150, # Aumentar un poco para respuestas más completas
    # Los modelos GPT-2 suelen ser menos coherentes con temperatura alta,
    # así que la ajustamos para mayor determinismo si queremos ver el efecto del prompt.
    temperature=0.7, # Ajustar para creatividad vs. coherencia
    do_sample=True,
    num_return_sequences=1,
    pad_token_id=tokenizer.eos_token_id # Para evitar warnings con gpt2
)
print("Modelo LLM cargado.")

print("\n--- Demostración de Prompting ---")

# --- Zero-shot Prompting ---
# Se le pide al modelo que realice una tarea sin ejemplos previos.
print("\n--- Zero-shot Prompting ---")
zero_shot_prompt = "Clasifica el sentimiento de la siguiente frase como Positivo, Negativo o Neutro: 'La película fue increíble, me encantó cada minuto.'"
print(f"Prompt (Zero-shot):\n{zero_shot_prompt}")
output = generator(zero_shot_prompt)
# Extraer solo la parte generada por el modelo
generated_zero_shot = output[0]['generated_text'][len(zero_shot_prompt):].strip()
print(f"Respuesta del modelo:\n{generated_zero_shot}")
print("-" * 30)

# --- Few-shot Prompting ---
# Se proporcionan algunos ejemplos para guiar al modelo en la tarea.
print("\n--- Few-shot Prompting ---")
few_shot_prompt = """
Clasifica el sentimiento de las siguientes frases:

Frase: "Este día es perfecto." -> Sentimiento: Positivo
Frase: "El servicio fue lento." -> Sentimiento: Negativo
Frase: "No está mal, pero podría mejorar." -> Sentimiento: Neutro
Frase: "La nueva actualización es fantástica y resuelve todos los problemas." -> Sentimiento:
"""
print(f"Prompt (Few-shot):\n{few_shot_prompt}")
output = generator(few_shot_prompt)
generated_few_shot = output[0]['generated_text'][len(few_shot_prompt):].strip()
print(f"Respuesta del modelo:\n{generated_few_shot}")
print("-" * 30)

# --- Chain-of-Thought Prompting ---
# Se le pide al modelo que razone paso a paso para llegar a la respuesta.
print("\n--- Chain-of-Thought Prompting ---")
cot_prompt = """
Resuelve el siguiente problema paso a paso.
Problema: Un tren sale de la estación A a las 9:00 AM y viaja a 60 km/h. Otro tren sale de la estación B (a 300 km de A) a las 10:00 AM y viaja a 40 km/h en dirección a la estación A. ¿A qué hora se encontrarán?

Paso 1: Calcular la distancia que recorre el primer tren antes de que el segundo salga.
"""
print(f"Prompt (Chain-of-Thought):\n{cot_prompt}")
output = generator(cot_prompt)
generated_cot = output[0]['generated_text'][len(cot_prompt):].strip()
print(f"Respuesta del modelo:\n{generated_cot}")
print("-" * 30)


# --- Instruction Prompting ---
# Se le dan instrucciones explícitas sobre cómo debe comportarse o qué tipo de respuesta debe dar.
print("\n--- Instruction Prompting ---")
instruction_prompt = """
Actúa como un asistente de escritura creativa. Genera una breve descripción para el inicio de una historia de fantasía que incluya un personaje principal en busca de un artefacto mágico en un bosque oscuro.
"""
print(f"Prompt (Instrucciones):\n{instruction_prompt}")
output = generator(instruction_prompt)
generated_instruction = output[0]['generated_text'][len(instruction_prompt):].strip()
print(f"Respuesta del modelo:\n{generated_instruction}")
print("-" * 30)

## Resumen

En este capítulo aprendimos:

- **Transformers**: Arquitectura basada en atención que genera representaciones contextuales.
- **Self-Attention**: Mecanismo que permite a cada token atender a todos los demás.
- **BERT**: Modelo encoder pre-entrenado bidireccionalmente, ideal para comprensión.
- **Fine-tuning**: Adaptación de modelos pre-entrenados a tareas específicas.
- **LLMs**: Modelos masivos con capacidades de generación, resumen y razonamiento.
- **Prompting**: Técnicas para comunicarse efectivamente con LLMs.

En el próximo capítulo exploraremos cómo integrar LLMs con herramientas externas para crear **Agentes de IA**.